In [ ]:
import numpy as np

from qiskit import QuantumCircuit
from qiskit.qasm3 import dumps
from qiskit.quantum_info import Operator
from qiskit.circuit.library import QFTGate

import json
import random
import math
import hashlib
from pathlib import Path

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ONE_QUBIT_GATES = [
    "x", "y", "z", "h", "s", "sdg", "t", "tdg",
    "rx", "ry", "rz",
]

TWO_QUBIT_GATES = ["cx", "cz", "swap"]

In [ ]:
def add_random_gate(qc, allow_two_qubit=True):
    n = qc.num_qubits

    choices = list(ONE_QUBIT_GATES)
    if allow_two_qubit and n >= 2:
        choices += TWO_QUBIT_GATES

    gate = random.choice(choices)

    if gate in TWO_QUBIT_GATES:
        a, b = random.sample(range(n), 2)
        getattr(qc, gate)(a, b)
        return

    q = random.randrange(n)

    if gate in ONE_QUBIT_GATES[:-3]:
        getattr(qc, gate)(q)
    else:
        theta = random.uniform(-math.pi, math.pi)
        getattr(qc, gate)(theta, q)


def append_random_computation(qc, num_gates):
    for _ in range(num_gates):
        add_random_gate(qc)


def copy_instruction(dst, instruction, src):
    qargs = [src.find_bit(q).index for q in instruction.qubits]
    cargs = [src.find_bit(c).index for c in instruction.clbits]
    dst.append(instruction.operation.copy(), qargs, cargs)

In [ ]:
def append_identity_block(qc, family=None):
    n = qc.num_qubits

    families = [
        "xx", "yy", "zz", "hh",
        "s_sdg", "sdg_s", "t_tdg", "tdg_t",
        "rx_inverse", "ry_inverse", "rz_inverse",
    ]

    if n >= 2:
        families += ["cx_cx", "cz_cz", "swap_swap"]

    family = family or random.choice(families)
    q = random.randrange(n)

    if family == "xx":
        qc.x(q); qc.x(q)
    elif family == "yy":
        qc.y(q); qc.y(q)
    elif family == "zz":
        qc.z(q); qc.z(q)
    elif family == "hh":
        qc.h(q); qc.h(q)
    elif family == "s_sdg":
        qc.s(q); qc.sdg(q)
    elif family == "sdg_s":
        qc.sdg(q); qc.s(q)
    elif family == "t_tdg":
        qc.t(q); qc.tdg(q)
    elif family == "tdg_t":
        qc.tdg(q); qc.t(q)
    elif family in {"rx_inverse", "ry_inverse", "rz_inverse"}:
        theta = random.uniform(-math.pi, math.pi)
        gate = family.split("_")[0]
        getattr(qc, gate)(theta, q)
        getattr(qc, gate)(-theta, q)
    elif family in {"cx_cx", "cz_cz", "swap_swap"}:
        a, b = random.sample(range(n), 2)
        gate = family.split("_")[0]
        getattr(qc, gate)(a, b)
        getattr(qc, gate)(a, b)
    else:
        raise ValueError(f"Unknown identity family: {family}")

    return family


def append_nested_identity_block(qc, family=None):
    n = qc.num_qubits

    if n < 2:
        return append_identity_block(qc)

    outer = random.choice(["cx", "cz", "swap"])
    a, b = random.sample(range(n), 2)

    getattr(qc, outer)(a, b)

    inner_family = family or random.choice([
        "xx", "yy", "zz", "hh",
        "s_sdg", "t_tdg",
        "rx_inverse", "ry_inverse", "rz_inverse",
    ])
    append_identity_block(qc, inner_family)

    getattr(qc, outer)(a, b)

    return f"{outer}_{inner_family}_{outer}"


def append_deep_nested_identity_block(qc, levels=None):
    n = qc.num_qubits

    if n < 2:
        return append_identity_block(qc)

    if levels is None:
        levels = random.randint(2, 5)

    if levels <= 1:
        return append_identity_block(qc)

    outer = random.choice(["cx", "cz", "swap"])
    a, b = random.sample(range(n), 2)
    getattr(qc, outer)(a, b)

    if levels == 2 or random.random() < 0.30:
        inner_rule = append_nested_identity_block(qc)
    else:
        inner_rule = append_deep_nested_identity_block(qc, levels=levels - 1)

    getattr(qc, outer)(a, b)

    return f"deep{levels}[{outer}({a},{b})::{inner_rule}::{outer}({a},{b})]"


def append_long_range_identity_block(qc):
    n = qc.num_qubits

    if n < 2:
        return append_identity_block(qc)

    outer = random.choice(["cx", "cz", "swap"])
    a, b = random.sample(range(n), 2)
    getattr(qc, outer)(a, b)

    rules = []
    for _ in range(random.randint(2, 6)):
        if random.random() < 0.30:
            rules.append(append_nested_identity_block(qc))
        else:
            rules.append(append_identity_block(qc))

    getattr(qc, outer)(a, b)

    return f"long_range[{outer}({a},{b}) + {' + '.join(rules)} + {outer}({a},{b})]"


def append_noise_block(qc, difficulty="mixed"):
    if difficulty == "easy":
        return append_identity_block(qc)
    if difficulty == "nested":
        return append_nested_identity_block(qc)
    if difficulty == "deep":
        return append_deep_nested_identity_block(qc, random.randint(3, 5))
    if difficulty == "long":
        return append_long_range_identity_block(qc)

    r = random.random()
    if r < 0.45:
        return append_identity_block(qc)
    if r < 0.70:
        return append_nested_identity_block(qc)
    if r < 0.88:
        return append_deep_nested_identity_block(qc)
    return append_long_range_identity_block(qc)

In [ ]:
def circuits_equal_up_to_global_phase(a, b, atol=1e-8):
    if a.num_qubits != b.num_qubits:
        return False

    ua = Operator(a).data
    ub = Operator(b).data
    relative = ua.conj().T @ ub

    diag = np.diag(relative)
    idx = np.argmax(np.abs(diag))
    phase = diag[idx]
    if abs(phase) < atol:
        return False
    phase /= abs(phase)

    return np.allclose(
        relative,
        phase * np.eye(relative.shape[0], dtype=complex),
        atol=atol,
    )

In [ ]:
def noisify_clean_circuit(
    clean,
    min_blocks=2,
    max_blocks=8,
    difficulty="mixed",
):
    noisy = QuantumCircuit(clean.num_qubits)
    rules = []

    for _ in range(random.randint(0, 2)):
        rules.append(append_noise_block(noisy, difficulty))

    total_blocks = random.randint(min_blocks, max_blocks)
    slots_left = total_blocks

    for inst in clean.data:
        if slots_left > 0 and random.random() < 0.55:
            count = random.randint(1, min(2, slots_left))
            for _ in range(count):
                rules.append(append_noise_block(noisy, difficulty))
                slots_left -= 1

        copy_instruction(noisy, inst, clean)

        if slots_left > 0 and random.random() < 0.45:
            rules.append(append_noise_block(noisy, difficulty))
            slots_left -= 1

    while slots_left > 0:
        rules.append(append_noise_block(noisy, difficulty))
        slots_left -= 1

    return noisy, clean.copy(), rules


def make_big_to_small_example(min_qubits=1, max_qubits=6):
    n = random.randint(min_qubits, max_qubits)
    target = QuantumCircuit(n)

    for _ in range(random.randint(0, 3)):
        add_random_gate(target)

    noisy, target, rules = noisify_clean_circuit(
        target,
        min_blocks=5,
        max_blocks=12,
        difficulty=random.choice(["mixed", "deep", "long"]),
    )

    return noisy, target, ["big_to_small"] + rules


def make_small_to_small_example(min_qubits=1, max_qubits=6):
    n = random.randint(min_qubits, max_qubits)
    target = QuantumCircuit(n)

    for _ in range(random.randint(1, 6)):
        add_random_gate(target)

    noisy, target, rules = noisify_clean_circuit(
        target,
        min_blocks=1,
        max_blocks=3,
        difficulty=random.choice(["easy", "nested", "mixed"]),
    )

    return noisy, target, ["small_to_small"] + rules


def make_big_to_big_example(min_qubits=2, max_qubits=6):
    n = random.randint(min_qubits, max_qubits)
    target = QuantumCircuit(n)

    num_useful = random.randint(10, 28)
    for _ in range(num_useful):
        add_random_gate(target)

    noisy, target, rules = noisify_clean_circuit(
        target,
        min_blocks=5,
        max_blocks=14,
        difficulty="mixed",
    )

    return noisy, target, ["big_to_big"] + rules


def make_ghz_target(n):
    qc = QuantumCircuit(n)
    qc.h(0)
    for q in range(n - 1):
        qc.cx(q, q + 1)
    return qc


def make_bell_pairs_target(n):
    qc = QuantumCircuit(n)

    for q in range(0, n - 1, 2):
        qc.h(q)
        qc.cx(q, q + 1)

    if n % 2 == 1:
        qc.h(n - 1)

    return qc


def make_entangling_backbone_target(n):
    qc = QuantumCircuit(n)

    for q in range(n):
        if random.random() < 0.55:
            qc.h(q)

    for _ in range(random.randint(n, 2 * n + 4)):
        a, b = random.sample(range(n), 2)
        gate = random.choice(["cx", "cz"])
        getattr(qc, gate)(a, b)

        # Interleave occasional meaningful rotations.
        if random.random() < 0.45:
            q = random.randrange(n)
            gate1 = random.choice(["rx", "ry", "rz"])
            getattr(qc, gate1)(random.uniform(-math.pi, math.pi), q)

    return qc


def make_qft_target(n):
    qc = QuantumCircuit(n)
    qc.append(QFTGate(n), range(n))
    return qc.decompose(reps=2)


def make_grover2_target():
    qc = QuantumCircuit(2)
    qc.h(0); qc.h(1)
    qc.cz(0, 1)
    qc.h(0); qc.h(1)
    qc.x(0); qc.x(1)
    qc.cz(0, 1)
    qc.x(0); qc.x(1)
    qc.h(0); qc.h(1)
    return qc


def make_structured_example():
    family = random.choices(
        ["ghz", "bell_pairs", "entangling", "qft", "grover2"],
        weights=[28, 20, 27, 15, 10],
        k=1,
    )[0]

    if family == "ghz":
        n = random.randint(3, 6)
        target = make_ghz_target(n)
    elif family == "bell_pairs":
        n = random.randint(2, 6)
        target = make_bell_pairs_target(n)
    elif family == "entangling":
        n = random.randint(3, 6)
        target = make_entangling_backbone_target(n)
    elif family == "qft":
        n = random.randint(2, 5)
        target = make_qft_target(n)
    elif family == "grover2":
        target = make_grover2_target()
    else:
        raise ValueError(family)

    noisy, target, rules = noisify_clean_circuit(
        target,
        min_blocks=3,
        max_blocks=10,
        difficulty="mixed",
    )

    return noisy, target, [f"structured:{family}"] + rules


def make_pure_identity_example(min_qubits=1, max_qubits=6):
    n = random.randint(min_qubits, max_qubits)
    noisy = QuantumCircuit(n)
    target = QuantumCircuit(n)
    rules = []

    for _ in range(random.randint(1, 7)):
        rules.append(append_noise_block(noisy, "mixed"))

    return noisy, target, rules


def make_deep_nested_example(min_qubits=2, max_qubits=6):
    n = random.randint(min_qubits, max_qubits)
    target = QuantumCircuit(n)

    for _ in range(random.randint(1, 8)):
        add_random_gate(target)

    return noisify_clean_circuit(
        target,
        min_blocks=2,
        max_blocks=7,
        difficulty=random.choice(["deep", "long", "mixed"]),
    )


def make_no_change_example(
    min_qubits=1,
    max_qubits=6,
    min_gates=1,
    max_gates=20,
):
    n = random.randint(min_qubits, max_qubits)
    qc = QuantumCircuit(n)
    append_random_computation(qc, random.randint(min_gates, max_gates))
    return qc, qc.copy(), ["no_change"]


In [ ]:
EXAMPLE_WEIGHTS = {
    "big_to_small": 12,
    "small_to_small": 10, 
    "big_to_big": 22,
    "structured": 24, # Well known algortihms like GHZ
    "deep_nested": 14, # Large amount of simplifications
    "pure_identity": 8, # Empty circuit as target
    "no_change": 10, # No optimization needed.
}


def make_example(kind=None):
    """Create one verified input/output pair."""
    if kind is None:
        kinds = list(EXAMPLE_WEIGHTS)
        weights = [EXAMPLE_WEIGHTS[k] for k in kinds]
        kind = random.choices(kinds, weights=weights, k=1)[0]

    if kind == "big_to_small":
        original, optimized, rules = make_big_to_small_example()
    elif kind == "small_to_small":
        original, optimized, rules = make_small_to_small_example()
    elif kind == "big_to_big":
        original, optimized, rules = make_big_to_big_example()
    elif kind == "structured":
        original, optimized, rules = make_structured_example()
    elif kind == "deep_nested":
        original, optimized, rules = make_deep_nested_example()
    elif kind == "pure_identity":
        original, optimized, rules = make_pure_identity_example()
    elif kind == "no_change":
        original, optimized, rules = make_no_change_example()
    else:
        raise ValueError(f"Unknown example kind: {kind}")

    # Never write an incorrect training pair
    if not circuits_equal_up_to_global_phase(original, optimized):
        raise RuntimeError(
            f"Generator bug: {kind} example is not unitary-equivalent. Rules={rules}"
        )

    input_qasm = dumps(original)
    output_qasm = dumps(optimized)

    return {
        "input": input_qasm,
        "output": output_qasm,
        "kind": kind,
        "rules": rules,
        "num_qubits": original.num_qubits,
        "input_gates": original.size(),
        "output_gates": optimized.size(),
        "input_depth": original.depth(),
        "output_depth": optimized.depth(),
        "gate_reduction": original.size() - optimized.size(),
        "depth_reduction": original.depth() - optimized.depth(),
        "reduction_ratio": (
            0.0 if original.size() == 0
            else (original.size() - optimized.size()) / original.size()
        ),
    }


In [ ]:
def create_dataset(
    num_examples,
    filename="quantum_optimization_dataset.jsonl",
    seed=42,
    deduplicate=True,
):
    random.seed(seed)
    np.random.seed(seed)

    path = Path(filename)
    seen = set()

    counts = {kind: 0 for kind in EXAMPLE_WEIGHTS}
    qubit_counts = {q: 0 for q in range(1, 7)}

    accepted = 0
    attempts = 0

    with path.open("w", encoding="utf-8") as f:
        while accepted < num_examples:
            attempts += 1
            example = make_example()

            if deduplicate:
                key = hashlib.sha256(
                    (
                        example["input"]
                        + "\n---TARGET---\n"
                        + example["output"]
                    ).encode("utf-8")
                ).hexdigest()

                if key in seen:
                    continue
                seen.add(key)

            f.write(json.dumps(example) + "\n")
            accepted += 1
            counts[example["kind"]] += 1
            qubit_counts[example["num_qubits"]] += 1

            if accepted % 250 == 0 or accepted == num_examples:
                print(f"{accepted}/{num_examples} examples generated")

    print(f"\nSaved dataset to: {path.resolve()}")
    print(f"Generation attempts: {attempts}")

    print("\nDataset composition:")
    for kind, count in counts.items():
        print(f"  {kind:16s}: {count:6d} ({count / num_examples:6.1%})")

    print("\nQubit-width composition:")
    for q, count in qubit_counts.items():
        print(f"  {q} qubits: {count:6d} ({count / num_examples:6.1%})")

    return counts, qubit_counts

In [ ]:
counts, qubit_counts = create_dataset(
    num_examples=5_000,
    filename="quantum_optimization_dataset.jsonl",
    seed=42,
)